# Jomana - BM25 Retrieval Milestone

**Assigned part:** Build BM25 retrieval for keyword-based lexical search.

This notebook loads the provided StackLite dataset, preprocesses the question corpus, builds an Okapi BM25 index, and returns the top-10 results for sample technical questions with citation-ready links.

## Scope

Covered here:

- Load and preprocess `data/stacklite_questions.csv`.
- Index question title, cleaned body text, and tags.
- Retrieve top-10 lexical BM25 matches for sample questions.
- Show source titles, snippets, scores, tags, and original Stack Exchange links.

Not covered in this milestone: semantic search, hybrid fusion, RAG generation, UI, or MAP/MRR/nDCG evaluation.

In [1]:
from pathlib import Path
import sys

import pandas as pd

# Works when the notebook is run from the repo root, the notebooks folder, or Colab /content.
PROJECT_CANDIDATES = [Path.cwd(), Path.cwd().parent, Path('/content')]
PROJECT_ROOT = next((path for path in PROJECT_CANDIDATES if (path / 'src').exists()), Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.bm25_retriever import BM25Retriever, load_stacklite_dataset, write_results_csv

DATASET_CANDIDATES = [
    PROJECT_ROOT / 'data/stacklite_questions.csv',
    Path.cwd() / 'data/stacklite_questions.csv',
    Path.cwd().parent / 'data/stacklite_questions.csv',
    Path('/content/data/stacklite_questions.csv'),
]
DATASET_PATH = next((path for path in DATASET_CANDIDATES if path.exists()), None)
if DATASET_PATH is None:
    raise FileNotFoundError('data/stacklite_questions.csv was not found. Run `dvc pull data/stacklite_questions.csv.dvc` first.')

print(f'Project root: {PROJECT_ROOT}')
print(f'Dataset path: {DATASET_PATH}')

Project root: C:\Users\RTX\Downloads\Advanced DL
Dataset path: C:\Users\RTX\Downloads\Advanced DL\data/stacklite_questions.csv


## 1. Load and Inspect the Corpus

Each StackLite question is treated as a searchable document. The citation fields are preserved so downstream RAG work can cite the original source.

In [2]:
documents = load_stacklite_dataset(DATASET_PATH)

summary = pd.DataFrame({
    'documents': [len(documents)],
    'sources': [', '.join(sorted({doc.source for doc in documents}))],
    'avg_title_length': [round(sum(len(doc.title.split()) for doc in documents) / len(documents), 2)],
    'answered_questions': [sum(1 for doc in documents if doc.answer_count > 0)],
})
summary

,documents,sources,avg_title_length,answered_questions
0,1500,"ai, datascience",9.66,1495


In [3]:
preview = pd.DataFrame([
    {
        'doc_id': doc.doc_id,
        'source': doc.source,
        'title': doc.title,
        'tags': ', '.join(doc.tags),
        'link': doc.link,
    }
    for doc in documents[:5]
])
preview

,doc_id,source,title,tags,link
0,ai:1768,ai,Could a paradox kill an AI?,"philosophy, decision-theory, mythology-of-ai, ...",https://ai.stackexchange.com/questions/1768/co...
1,ai:35,ai,What is the difference between artificial inte...,"machine-learning, comparison, terminology, ai-...",https://ai.stackexchange.com/questions/35/what...
2,ai:111,ai,How could self-driving cars make ethical decis...,"philosophy, ethics, autonomous-vehicles, decis...",https://ai.stackexchange.com/questions/111/how...
3,ai:2008,ai,How can neural networks deal with varying inpu...,"neural-networks, machine-learning, deep-learni...",https://ai.stackexchange.com/questions/2008/ho...
4,ai:1479,ai,Do scientists know what is happening inside ar...,"neural-networks, deep-learning, convolutional-...",https://ai.stackexchange.com/questions/1479/do...


## 2. Build the BM25 Index

BM25 is a strong keyword baseline because it rewards exact term overlap while normalizing for document length. The index uses:

- `k1 = 1.5`
- `b = 0.75`
- IDF smoothing: `log(1 + (N - df + 0.5) / (df + 0.5))`

In [4]:
bm25 = BM25Retriever(k1=1.5, b=0.75).fit(documents)

print(f'Indexed documents: {len(bm25.documents):,}')
print(f'Vocabulary size: {len(bm25.idf):,}')
print(f'Average document length: {bm25.avg_doc_length:.2f} tokens')

Indexed documents: 1,500
Vocabulary size: 13,242
Average document length: 84.92 tokens


## 3. Retrieve Top-10 Results for Sample Questions

The sample questions are phrased like real user questions and cover both AI and data-science topics present in the corpus.

In [5]:
sample_queries = [
    'Why are micro average precision recall and F1 equal in multiclass classification?',
    'What is the difference between artificial intelligence and machine learning?',
    'What are deconvolutional layers in convolutional neural networks?',
    'How do I set class weights for imbalanced classes in Keras?',
    'What is the dying ReLU problem in neural networks?',
]

query_to_results = {query: bm25.search(query, top_k=10) for query in sample_queries}

for query, results in query_to_results.items():
    print(f'\nQUERY: {query}')
    display(pd.DataFrame(results)[[
        'rank', 'score', 'source', 'title', 'tags', 'snippet', 'link'
    ]])


QUERY: Why are micro average precision recall and F1 equal in multiclass classification?

QUERY: What is the difference between artificial intelligence and machine learning?

QUERY: What are deconvolutional layers in convolutional neural networks?

QUERY: How do I set class weights for imbalanced classes in Keras?

QUERY: What is the dying ReLU problem in neural networks?


,rank,score,source,title,tags,snippet,link
0,1,48.493982,datascience,Micro Average vs Macro average Performance in ...,"multiclass-classification, model-evaluations",Micro Average vs Macro average Performance in ...,https://datascience.stackexchange.com/question...
1,2,30.747687,datascience,Why is the F-measure preferred for classificat...,"machine-learning, model-evaluations, scoring, ...",Why is the F-measure preferred for classificat...,https://datascience.stackexchange.com/question...
2,3,24.953232,datascience,What's the difference between Sklearn F1 score...,"classification, scikit-learn, metric",What's the difference between Sklearn F1 score...,https://datascience.stackexchange.com/question...
3,4,24.292095,datascience,How to calculate mAP for detection task for th...,"machine-learning, neural-network, svm, compute...",...ulate mAP for detection task for the PASCAL...,https://datascience.stackexchange.com/question...
4,5,22.189749,datascience,"How to get accuracy, F1, precision and recall,...","machine-learning, neural-network, deep-learnin...","How to get accuracy, F1, precision and recall,...",https://datascience.stackexchange.com/question...
5,6,19.732273,datascience,How to interpret classification report of scik...,"classification, metric, binary",How to interpret classification report of scik...,https://datascience.stackexchange.com/question...
6,7,19.564670,datascience,macro average and weighted average meaning in ...,"classification, accuracy, class-imbalance",macro average and weighted average meaning in ...,https://datascience.stackexchange.com/question...
7,8,18.118757,datascience,When is precision more important over recall?,"machine-learning, model-evaluations",When is precision more important over recall? ...,https://datascience.stackexchange.com/question...
8,9,15.495841,datascience,Unbalanced multiclass data with XGBoost,"classification, xgboost, multiclass-classifica...",Unbalanced multiclass data with XGBoost Unbala...,https://datascience.stackexchange.com/question...
9,10,14.826174,datascience,In XGBoost would we evaluate results with a Pr...,xgboost,In XGBoost would we evaluate results with a Pr...,https://datascience.stackexchange.com/question...


,rank,score,source,title,tags,snippet,link
0,1,22.033671,datascience,Difference between machine learning and artifi...,"machine-learning, ai, theory",Difference between machine learning and artifi...,https://datascience.stackexchange.com/question...
1,2,18.972563,ai,What is the difference between artificial inte...,"machine-learning, comparison, terminology, ai-...",What is the difference between artificial inte...,https://ai.stackexchange.com/questions/35/what...
2,3,16.138720,ai,How is Bayes' Theorem used in artificial intel...,"machine-learning, applications, bayes-theorem",How is Bayes' Theorem used in artificial intel...,https://ai.stackexchange.com/questions/2738/ho...
3,4,15.919972,ai,What are the top artificial intelligence journ...,research,What are the top artificial intelligence journ...,https://ai.stackexchange.com/questions/2306/wh...
4,5,15.676151,ai,What is the difference between artificial inte...,"terminology, robotics, robots, comparison",What is the difference between artificial inte...,https://ai.stackexchange.com/questions/1462/wh...
5,6,15.113489,ai,What is the difference between artificial inte...,"terminology, comparison, cognitive-science",What is the difference between artificial inte...,https://ai.stackexchange.com/questions/1847/wh...
6,7,14.658173,ai,What are examples of techniques to prevent bia...,"machine-learning, datasets, social, ai-safety,...",What are examples of techniques to prevent bia...,https://ai.stackexchange.com/questions/3175/wh...
7,8,14.165380,ai,Is artificial intelligence really just human i...,"philosophy, intelligence, artificial-creativity",Is artificial intelligence really just human i...,https://ai.stackexchange.com/questions/16646/i...
8,9,13.934380,ai,What is the difference between artificial inte...,"comparison, terminology, definitions",What is the difference between artificial inte...,https://ai.stackexchange.com/questions/7446/wh...
9,10,13.650550,ai,"What is ""early stopping"" in machine learning?","deep-learning, definitions, overfitting, regul...","What is ""early stopping"" in machine learning? ...",https://ai.stackexchange.com/questions/16/what...


,rank,score,source,title,tags,snippet,link
0,1,22.192062,datascience,What are deconvolutional layers?,"neural-network, convolutional-neural-network, ...",What are deconvolutional layers? What are deco...,https://datascience.stackexchange.com/question...
1,2,17.607478,datascience,Why do convolutional neural networks work?,"machine-learning, deep-learning, neural-networ...",Why do convolutional neural networks work? Why...,https://datascience.stackexchange.com/question...
2,3,13.970970,ai,Do deep learning algorithms represent ensemble...,"neural-networks, deep-learning, convolutional-...",...t to model high-level abstractions in data ...,https://ai.stackexchange.com/questions/1978/do...
3,4,12.967898,ai,What is a fully convolution network?,"machine-learning, convolutional-neural-network...",...rk? What is a fully convolution network? I ...,https://ai.stackexchange.com/questions/21810/w...
4,5,12.577344,ai,When should I use 3D convolutions?,"convolutional-neural-networks, reference-reque...",When should I use 3D convolutions? When should...,https://ai.stackexchange.com/questions/13692/w...
5,6,12.541757,datascience,What are the differences between Convolutional...,"machine-learning, neural-network, deep-learnin...",What are the differences between Convolutional...,https://datascience.stackexchange.com/question...
6,7,12.071487,ai,Why aren't there neural networks that connect ...,"neural-networks, feedforward-neural-networks, ...",Why aren't there neural networks that connect ...,https://ai.stackexchange.com/questions/5862/wh...
7,8,12.019565,ai,How to handle rectangular images in convolutio...,"neural-networks, convolutional-neural-networks...",How to handle rectangular images in convolutio...,https://ai.stackexchange.com/questions/8323/ho...
8,9,11.856026,datascience,How to set the number of neurons and layers in...,"machine-learning, neural-network, deep-learnin...",How to set the number of neurons and layers in...,https://datascience.stackexchange.com/question...
9,10,11.710486,ai,Which layer in a CNN consumes more training ti...,"neural-networks, deep-learning, convolutional-...",Which layer in a CNN consumes more training ti...,https://ai.stackexchange.com/questions/7865/wh...


,rank,score,source,title,tags,snippet,link
0,1,32.996294,datascience,How to set class weights for imbalanced classe...,"deep-learning, classification, keras, weighted...",How to set class weights for imbalanced classe...,https://datascience.stackexchange.com/question...
1,2,23.280822,datascience,Deep network not able to learn imbalanced data...,"deep-learning, keras, tensorflow, multiclass-c...",Deep network not able to learn imbalanced data...,https://datascience.stackexchange.com/question...
2,3,18.837304,datascience,Macro- or micro-average for imbalanced class p...,"machine-learning, model-evaluations, class-imb...",Macro- or micro-average for imbalanced class p...,https://datascience.stackexchange.com/question...
3,4,17.742770,datascience,What loss function to use for imbalanced class...,"neural-network, pytorch",What loss function to use for imbalanced class...,https://datascience.stackexchange.com/question...
4,5,14.879306,datascience,Why doesn't class weight resolve the imbalance...,"classification, class-imbalance, weighted-data",Why doesn't class weight resolve the imbalance...,https://datascience.stackexchange.com/question...
5,6,14.612408,ai,"How to implement an ""unknown"" class in multi-c...","deep-learning, classification, image-recogniti...","How to implement an ""unknown"" class in multi-c...",https://ai.stackexchange.com/questions/4889/ho...
6,7,14.427566,datascience,How does Keras calculate accuracy?,"neural-network, deep-learning, keras",How does Keras calculate accuracy? How does Ke...,https://datascience.stackexchange.com/question...
7,8,13.442460,datascience,Quick guide into training highly imbalanced da...,"machine-learning, classification, dataset, cla...",Quick guide into training highly imbalanced da...,https://datascience.stackexchange.com/question...
8,9,13.247156,datascience,One-Class discriminatory classification with i...,"machine-learning, data-mining, python, classif...",One-Class discriminatory classification with i...,https://datascience.stackexchange.com/question...
9,10,13.134933,datascience,Unbalanced multiclass data with XGBoost,"classification, xgboost, multiclass-classifica...",Unbalanced multiclass data with XGBoost Unbala...,https://datascience.stackexchange.com/question...


,rank,score,source,title,tags,snippet,link
0,1,24.127748,datascience,"What is the ""dying ReLU"" problem in neural net...","machine-learning, neural-network, deep-learning","What is the ""dying ReLU"" problem in neural net...",https://datascience.stackexchange.com/question...
1,2,21.053690,datascience,How to check for dead relu neurons,"machine-learning, deep-learning, neural-networ...",How to check for dead relu neurons How to chec...,https://datascience.stackexchange.com/question...
2,3,10.786410,ai,What happens when I mix activation functions?,"neural-networks, machine-learning, activation-...",...happens when I mix activation functions? Th...,https://ai.stackexchange.com/questions/9828/wh...
3,4,9.889884,datascience,Difference of Activation Functions in Neural N...,"neural-network, activation-function",Difference of Activation Functions in Neural N...,https://datascience.stackexchange.com/question...
4,5,9.754422,ai,Can neural networks efficiently solve the trav...,"neural-networks, reference-request, time-compl...",Can neural networks efficiently solve the trav...,https://ai.stackexchange.com/questions/2874/ca...
5,6,9.706955,datascience,How does multicollinearity affect neural netwo...,"neural-network, correlation",How does multicollinearity affect neural netwo...,https://datascience.stackexchange.com/question...
6,7,9.369132,ai,Can prior knowledge be encoded in deep neural ...,"neural-networks, deep-learning, deep-neural-ne...",Can prior knowledge be encoded in deep neural ...,https://ai.stackexchange.com/questions/5322/ca...
7,8,9.277289,ai,Are there neural networks with very few nodes ...,neural-networks,Are there neural networks with very few nodes ...,https://ai.stackexchange.com/questions/10944/a...
8,9,8.824567,datascience,Deep Neural Network - Backpropogation with ReLU,"neural-network, backpropagation",Deep Neural Network - Backpropogation with ReL...,https://datascience.stackexchange.com/question...
9,10,8.695480,ai,What tools are used to deal with adversarial e...,"resource-request, adversarial-ml, ai-safety, a...",What tools are used to deal with adversarial e...,https://ai.stackexchange.com/questions/6892/wh...


## 4. Result Quality Summary

The tables above show the full top-10 rankings. The compact table below highlights the top-1 result for each sample question, which is the fastest way to judge whether the BM25 baseline is retrieving the expected source document.

In [6]:
top1_summary = pd.DataFrame([
    {
        'sample_question': query,
        'top_1_title': results[0]['title'],
        'source': results[0]['source'],
        'bm25_score': results[0]['score'],
        'citation_link': results[0]['link'],
    }
    for query, results in query_to_results.items()
])

top1_summary[['sample_question', 'top_1_title', 'source', 'bm25_score', 'citation_link']]

,sample_question,top_1_title,source,bm25_score,citation_link
0,Why are micro average precision recall and F1 ...,Micro Average vs Macro average Performance in ...,datascience,48.493982,https://datascience.stackexchange.com/question...
1,What is the difference between artificial inte...,Difference between machine learning and artifi...,datascience,22.033671,https://datascience.stackexchange.com/question...
2,What are deconvolutional layers in convolution...,What are deconvolutional layers?,datascience,22.192062,https://datascience.stackexchange.com/question...
3,How do I set class weights for imbalanced clas...,How to set class weights for imbalanced classe...,datascience,32.996294,https://datascience.stackexchange.com/question...
4,What is the dying ReLU problem in neural netwo...,"What is the ""dying ReLU"" problem in neural net...",datascience,24.127748,https://datascience.stackexchange.com/question...


### Result Quality Observations

- The BM25 baseline retrieves a highly relevant StackLite question at rank 1 for each sample question.
- The strongest results happen when the user query contains distinctive technical terms such as `micro average`, `deconvolutional layers`, `class weights`, `Keras`, or `dying ReLU`.
- Each result preserves citation-ready metadata: title, original Stack Exchange link, tags, source collection, score, answer count, and a matched snippet.
- The title is lightly boosted during indexing because Stack Exchange titles usually contain the cleanest summary of the technical question.
- Main limitation: BM25 is lexical, so it can miss synonym-heavy queries where the wording differs from the source document. That limitation is exactly why the next teammate adds semantic retrieval and hybrid fusion.
- This section is a qualitative sanity check only. Formal MAP@10, MRR@10, and nDCG@10 evaluation belongs to the evaluation milestone.

## 5. Save Results for Reporting

The CSV gives the team a reusable artifact for the report, slides, or later comparison with semantic retrieval.

In [7]:
output_path = PROJECT_ROOT / 'results' / 'bm25_sample_top10.csv'
write_results_csv(query_to_results, output_path)
print(f'Saved: {output_path}')

Saved: C:\Users\RTX\Downloads\Advanced DL\results\bm25_sample_top10.csv


## Handoff Notes

This milestone produces a clean BM25 baseline. The next project parts can reuse `BM25Retriever.search(query, top_k=10)` as the lexical retrieval component for evaluation, semantic comparison, hybrid fusion, and RAG citation grounding.